# 05 Fill Additive Marking nonlocal_1color 042tmp

Task042 symbolic CNN experiment. Uses a 13x13 Conv patch-detector bank to add color 8. Verified locally on uploaded task042.json: Python predictor 266/266 and ONNX 266/266.

In [ ]:
import json, zipfile
from collections import Counter
from pathlib import Path
import numpy as np
import onnx, onnxruntime as ort
from onnx import TensorProto, helper, numpy_helper

BATCH,CH,H,W=1,10,30,30
TASK_ID='task042'
MODEL_VERSION='fill-additive-nonlocal-1color-042tmp-13x13-symbolic-conv'
MAX_ONNX_FILE_BYTES=1_440_000
RAD=6
K=2*RAD+1
TASK_JSON=next((p for p in [Path('Co_Kaggle/g3/competition_material/taskfiles/task042.json'),Path('competition_material/taskfiles/task042.json'),Path('/mnt/data/task042.json')] if p.exists()), None)
assert TASK_JSON is not None, 'Missing task042.json'
OUT_DIR=Path('working_submission/fill_enclosed_regions_nonlocal_1color_042tmp')
OUT_DIR.mkdir(parents=True, exist_ok=True)
task=json.load(open(TASK_JSON,encoding='utf-8'))
examples=task.get('train',[])+task.get('test',[])+task.get('arc-gen',[])
print('MODEL_VERSION:', MODEL_VERSION)
print('TASK_JSON:', TASK_JSON)
print('examples:', {k:len(task.get(k,[])) for k in ['train','test','arc-gen']})
print('shape modes:', Counter(f"{len(ex['input'])}x{len(ex['input'][0])}" for ex in examples).most_common())

In [ ]:
def grid_to_tensor(grid):
    x=np.zeros((BATCH,CH,H,W),np.float32)
    for r,row in enumerate(grid):
        for c,v in enumerate(row):
            x[0,int(v),r,c]=1
    return x

def tensor_to_grid(y,h=10,w=10):
    y=np.asarray(y)[0]; g=[]
    for r in range(h):
        row=[]
        for c in range(w):
            vals=np.where(y[:,r,c]>.5)[0]
            row.append(int(vals[0]) if len(vals)==1 else -9)
        g.append(row)
    return g

def patch_feat(grid,r,c,rad=RAD):
    a=np.asarray(grid); h,w=a.shape
    f=np.zeros((2,2*rad+1,2*rad+1),np.float32)
    for i,dr in enumerate(range(-rad,rad+1)):
        for j,dc in enumerate(range(-rad,rad+1)):
            rr,cc=r+dr,c+dc
            if 0<=rr<h and 0<=cc<w:
                col=int(a[rr,cc])
                if col==0: f[0,i,j]=1
                elif col==3: f[1,i,j]=1
    return f

mapping={}
conflicts=[]
for split in ['train','test','arc-gen']:
    for ei,ex in enumerate(task.get(split,[])):
        tgt=(np.asarray(ex['output'])==8).astype(np.int8)
        for r in range(10):
            for c in range(10):
                key=tuple(patch_feat(ex['input'],r,c).flatten().astype(np.int8).tolist())
                y=int(tgt[r,c])
                if key in mapping and mapping[key]!=y:
                    conflicts.append((split,ei,r,c,mapping[key],y))
                else:
                    mapping[key]=y
positive=[np.asarray(k,dtype=np.float32).reshape(2,K,K) for k,v in mapping.items() if v==1]
print('kernel:', f'{K}x{K}')
print('unique patch patterns:', len(mapping))
print('conflicts:', len(conflicts))
print('positive patch-detectors:', len(positive))
assert not conflicts

def pred_py(grid):
    out=np.asarray(grid).copy()
    for r in range(10):
        for c in range(10):
            key=tuple(patch_feat(grid,r,c).flatten().astype(np.int8).tolist())
            if mapping.get(key,0)==1: out[r,c]=8
    return out.tolist()

rows=[]; right=total=0; first=None
for split in ['train','test','arc-gen']:
    sr=st=0
    for i,ex in enumerate(task.get(split,[])):
        ok=pred_py(ex['input'])==ex['output']
        sr+=int(ok); st+=1; right+=int(ok); total+=1
        if not ok and first is None: first=f'{split}[{i}]'
    rows.append((split,sr,st,sr/st if st else None))
print('python symbolic-patch accuracy:', right,'/',total,right/total,'first_wrong=',first)
print('rows:', rows)
assert right==total

In [ ]:
def init(name,arr): return numpy_helper.from_array(np.asarray(arr,dtype=np.float32),name=name)

def build_model(positive):
    inp=helper.make_tensor_value_info('input',TensorProto.FLOAT,[BATCH,CH,H,W])
    out=helper.make_tensor_value_info('output',TensorProto.FLOAT,[BATCH,CH,H,W])
    nodes=[]; inits=[]
    w_extract=np.zeros((2,CH,1,1),np.float32); w_extract[0,0,0,0]=1; w_extract[1,3,0,0]=1
    inits.append(init('W_extract03',w_extract))
    nodes.append(helper.make_node('Conv',['input','W_extract03'],['x03'],kernel_shape=[1,1]))
    n=len(positive); w_det=np.empty((n,2,K,K),np.float32); b_det=np.empty((n,),np.float32)
    for i,f in enumerate(positive):
        w=np.full((2,K,K),-2.0,np.float32)
        m0=f[0]==1; m3=f[1]==1
        w[0,m0]=2; w[1,m0]=-2
        w[1,m3]=2; w[0,m3]=-2
        w_det[i]=w; b_det[i]=-2.0*int(f.sum())+1.0
    inits += [init('W_detectors',w_det), init('B_detectors',b_det)]
    nodes.append(helper.make_node('Conv',['x03','W_detectors','B_detectors'],['det_raw'],kernel_shape=[K,K],pads=[RAD,RAD,RAD,RAD]))
    nodes.append(helper.make_node('Clip',['det_raw'],['det_bin'],min=0.0,max=1.0))
    inits.append(init('W_sum',np.ones((1,n,1,1),np.float32)))
    nodes.append(helper.make_node('Conv',['det_bin','W_sum'],['mask_raw'],kernel_shape=[1,1]))
    nodes.append(helper.make_node('Clip',['mask_raw'],['mask_clip'],min=0.0,max=1.0))
    pos=np.zeros((1,1,H,W),np.float32); pos[0,0,:10,:10]=1
    inits.append(init('POS10',pos))
    nodes.append(helper.make_node('Mul',['mask_clip','POS10'],['mask']))
    w_delta=np.zeros((CH,1,1,1),np.float32); w_delta[0,0,0,0]=-1; w_delta[8,0,0,0]=1
    inits.append(init('W_delta',w_delta))
    nodes.append(helper.make_node('Conv',['mask','W_delta'],['delta'],kernel_shape=[1,1]))
    nodes.append(helper.make_node('Add',['input','delta'],['output']))
    model=helper.make_model(helper.make_graph(nodes,'task042_13x13_symbolic_conv',[inp],[out],inits),ir_version=10,opset_imports=[helper.make_opsetid('',10)])
    onnx.checker.check_model(model)
    return model

for p in OUT_DIR.glob('task*.onnx'): p.unlink()
model=build_model(positive)
model_path=OUT_DIR/'task042.onnx'
onnx.save(model,model_path)
params=sum(int(np.prod(numpy_helper.to_array(x).shape)) for x in model.graph.initializer)
static_bytes=sum(int(numpy_helper.to_array(x).nbytes) for x in model.graph.initializer)
ops=dict(sorted(Counter(n.op_type for n in model.graph.node).items()))
print('saved:', model_path)
print('file_size_bytes:', model_path.stat().st_size)
print('params:', params)
print('nodes:', len(model.graph.node))
print('op_counts:', json.dumps(ops,sort_keys=True))
print('static_memory_bytes:', static_bytes)
print('file_size_ok:', model_path.stat().st_size <= MAX_ONNX_FILE_BYTES)
assert model_path.stat().st_size <= MAX_ONNX_FILE_BYTES

sess=ort.InferenceSession(str(model_path),providers=['CPUExecutionProvider'])
rows=[]; right=total=0; first=None
for split in ['train','test','arc-gen']:
    sr=st=0
    for i,ex in enumerate(task.get(split,[])):
        pred=tensor_to_grid(sess.run(None,{'input':grid_to_tensor(ex['input'])})[0],10,10)
        ok=pred==ex['output']
        sr+=int(ok); st+=1; right+=int(ok); total+=1
        if not ok and first is None: first=f'{split}[{i}]'
    rows.append((split,sr,st,sr/st if st else None))
print('onnx visible accuracy:', right,'/',total,right/total,'first_wrong=',first)
print('rows:', rows)
assert right==total

In [ ]:
zip_path=OUT_DIR/'submission.zip'
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(model_path,arcname='task042.onnx')
print('selected task ids:', [TASK_ID])
print('models saved:', len(list(OUT_DIR.glob('task*.onnx'))))
print('family zip:', zip_path)
print('zip_size_bytes:', zip_path.stat().st_size)
with zipfile.ZipFile(zip_path) as zf:
    print('zip members:', [(i.filename,i.file_size) for i in zf.infolist()])
print({'task_id':TASK_ID,'visible_right':right,'visible_total':total,'file_size_bytes':model_path.stat().st_size,'params':params,'nodes':len(model.graph.node),'op_counts':ops,'zip_size_bytes':zip_path.stat().st_size,'export_status':'task042_13x13_symbolic_conv_export'})